# 04 — Model Training & Hyperparameter Tuning

**Project #8 — Throughput Prediction in a Dense 5G deployment with Transfer Learning**  
Network Data Analysis Laboratory 2025-2026, Politecnico di Milano

---

## Goals
1. Train **Neural Network (MLP)** and **Random Forest** regressors on ACC Arena data
2. Perform **hyperparameter tuning** with cross-validation (Optuna for RF, grid/Optuna for NN)
3. Experiment with different values of **X** (number of closest-user neighbours)
4. Save the best model checkpoints to `results/models/`

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
import json

import torch

from src.models.neural_network import MLPRegressor, train_mlp, predict_mlp, DEVICE
from src.models.random_forest  import RFRegressor
from src.utils.metrics import evaluate, print_metrics
from src.utils.visualization import plot_training_history

PROCESSED_DIR = Path('../data/processed')
MODELS_DIR    = Path('../results/models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

X_VALUES = [1, 3, 5, 10]  # neighbour count

print(f"Using device: {DEVICE}")

## 1. Load preprocessed data

In [ ]:
def load_splits(prefix):
    """Load train/val/test arrays from a common filename prefix."""
    return (
        np.load(f'{prefix}_X_train.npy'),
        np.load(f'{prefix}_X_val.npy'),
        np.load(f'{prefix}_X_test.npy'),
        np.load(f'{prefix}_y_train.npy'),
        np.load(f'{prefix}_y_val.npy'),
        np.load(f'{prefix}_y_test.npy'),
    )

# No-neighbour baseline
X_train_base, X_val_base, X_test_base, y_train, y_val, y_test = load_splits(
    PROCESSED_DIR / 'acc'
)

# With neighbours
splits = {x: load_splits(PROCESSED_DIR / f'acc_X{x}') for x in X_VALUES}

print(f"Baseline feature dim : {X_train_base.shape[1]}")
for x, (Xtr, *_) in splits.items():
    print(f"X={x:2d} feature dim       : {Xtr.shape[1]}")

## 2. Baseline: Random Forest (no neighbour features)

In [ ]:
# 2a. Quick RF baseline with default params
rf_base = RFRegressor(random_state=RANDOM_SEED)
rf_base.fit(X_train_base, y_train)
y_pred_rf_base = rf_base.predict(X_test_base)

metrics_rf_base = evaluate(y_test, y_pred_rf_base, training_time_s=rf_base.training_time_s)
print_metrics(metrics_rf_base, 'RF Baseline (no neighbours)')

In [ ]:
# 2b. RF hyperparameter tuning (Optuna, 5-fold CV)
rf_tuned = RFRegressor(random_state=RANDOM_SEED)
best_rf_params = rf_tuned.tune(X_train_base, y_train, n_trials=50, cv=5)

print('Best RF hyperparameters:')
print(best_rf_params)

# Re-fit with best params on full training set
rf_tuned.fit(X_train_base, y_train)
y_pred_rf_tuned = rf_tuned.predict(X_test_base)
metrics_rf_tuned = evaluate(y_test, y_pred_rf_tuned, training_time_s=rf_tuned.training_time_s)
print_metrics(metrics_rf_tuned, 'RF Tuned (no neighbours)')

In [ ]:
# Save tuned RF
rf_tuned.save(MODELS_DIR / 'rf_acc_arena_base.pkl')
print('RF saved.')

## 3. Baseline: Neural Network (no neighbour features)

In [ ]:
input_dim = X_train_base.shape[1]

mlp_base = MLPRegressor(
    input_dim=input_dim,
    hidden_dims=[256, 128, 64],
    dropout=0.2,
)

history_base = train_mlp(
    mlp_base,
    X_train_base, y_train,
    X_val_base,   y_val,
    lr=1e-3,
    max_epochs=100,
    patience=10,
    batch_size=512,
    checkpoint_path=MODELS_DIR / 'mlp_acc_arena_base.pt',
)

fig = plot_training_history(history_base, model_name='MLP_base', save=True)
plt.show()

In [ ]:
y_pred_mlp_base = predict_mlp(mlp_base, X_test_base)
metrics_mlp_base = evaluate(
    y_test, y_pred_mlp_base,
    training_time_s=history_base['training_time_s'],
)
print_metrics(metrics_mlp_base, 'MLP Baseline (no neighbours)')

## 4. NN Hyperparameter Tuning (Optuna)

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_nn(trial):
    hidden_dims = [
        trial.suggest_categorical(f'h{i}', [64, 128, 256, 512])
        for i in range(trial.suggest_int('n_layers', 2, 4))
    ]
    dropout    = trial.suggest_float('dropout',      0.0, 0.5)
    lr         = trial.suggest_float('lr',           1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch',  [256, 512, 1024])

    model = MLPRegressor(input_dim=input_dim, hidden_dims=hidden_dims, dropout=dropout)
    hist  = train_mlp(
        model, X_train_base, y_train, X_val_base, y_val,
        lr=lr, batch_size=batch_size, max_epochs=50, patience=8, verbose=False,
    )
    return min(hist['val_loss'])   # minimise val MSE

study_nn = optuna.create_study(direction='minimize')
study_nn.optimize(objective_nn, n_trials=30, show_progress_bar=True)

best_nn_params = study_nn.best_params
print('\nBest NN hyperparameters:')
print(best_nn_params)

In [ ]:
# Retrain with best hyperparameters
best_hidden_dims = [
    best_nn_params[f'h{i}']
    for i in range(best_nn_params['n_layers'])
]

mlp_tuned = MLPRegressor(
    input_dim=input_dim,
    hidden_dims=best_hidden_dims,
    dropout=best_nn_params['dropout'],
)

history_tuned = train_mlp(
    mlp_tuned,
    X_train_base, y_train, X_val_base, y_val,
    lr=best_nn_params['lr'],
    batch_size=best_nn_params['batch'],
    max_epochs=150, patience=15,
    checkpoint_path=MODELS_DIR / 'mlp_acc_arena_tuned_base.pt',
)

y_pred_mlp_tuned = predict_mlp(mlp_tuned, X_test_base)
metrics_mlp_tuned = evaluate(y_test, y_pred_mlp_tuned, history_tuned['training_time_s'])
print_metrics(metrics_mlp_tuned, 'MLP Tuned (no neighbours)')

fig = plot_training_history(history_tuned, model_name='MLP_tuned_base', save=True)
plt.show()

## 5. Experiment with X closest-user features

In [ ]:
results_by_x = {}   # {x_value: {'rf': metrics, 'mlp': metrics}}

for x_val in X_VALUES:
    X_tr, X_v, X_te, y_tr, y_v, y_te = splits[x_val]
    feat_dim = X_tr.shape[1]
    print(f"\n{'='*50}")
    print(f" X = {x_val}  (feature dim: {feat_dim})")
    print('='*50)

    # --- Random Forest ---
    rf_x = RFRegressor(random_state=RANDOM_SEED)
    rf_x.fit(X_tr, y_tr)
    preds_rf = rf_x.predict(X_te)
    m_rf = evaluate(y_te, preds_rf, rf_x.training_time_s)
    rf_x.save(MODELS_DIR / f'rf_acc_arena_X{x_val}.pkl')
    print_metrics(m_rf, f'RF  X={x_val}')

    # --- MLP ---
    mlp_x = MLPRegressor(
        input_dim=feat_dim,
        hidden_dims=best_hidden_dims,
        dropout=best_nn_params['dropout'],
    )
    hist_x = train_mlp(
        mlp_x, X_tr, y_tr, X_v, y_v,
        lr=best_nn_params['lr'],
        batch_size=best_nn_params['batch'],
        max_epochs=150, patience=15,
        checkpoint_path=MODELS_DIR / f'mlp_acc_arena_X{x_val}.pt',
    )
    preds_mlp = predict_mlp(mlp_x, X_te)
    m_mlp = evaluate(y_te, preds_mlp, hist_x['training_time_s'])
    print_metrics(m_mlp, f'MLP X={x_val}')

    results_by_x[x_val] = {'rf': m_rf, 'mlp': m_mlp}

print('\nAll X-value experiments complete.')

## 6. Save training metadata

In [ ]:
# Serialise results for notebook 05
import json

all_results = {
    'baseline': {
        'rf_tuned':  metrics_rf_tuned,
        'mlp_tuned': metrics_mlp_tuned,
    },
    'by_x': {str(x): v for x, v in results_by_x.items()},
    'best_nn_params': best_nn_params,
    'best_rf_params': best_rf_params,
}

with open(MODELS_DIR / 'training_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print('Training results saved to results/models/training_results.json')